In [12]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2


def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c


print("Libraries and Haversine function loaded successfully.")

Libraries and Haversine function loaded successfully.


In [13]:
mobility_spatial = pd.read_csv("mobility_store_proximity.csv")
store_analysis = pd.read_csv("store_spatial_analysis.csv")
stores_df = pd.read_csv("stores.csv")

print("Mobility spatial data:", mobility_spatial.shape)
print("Store analysis data:", store_analysis.shape)
print("Stores data:", stores_df.shape)

print("\nAll datasets loaded successfully.")

Mobility spatial data: (5000, 22)
Store analysis data: (10, 13)
Stores data: (10, 7)

All datasets loaded successfully.


In [14]:
store_metrics = (
    mobility_spatial
    .groupby(["nearest_store_id", "nearest_store_name"])
    .agg(
        total_mobility_pings=("ping_id", "count"),
        avg_distance_km=("distance_to_nearest_store_km", "mean"),
        pings_within_500m=("within_500m", "sum"),
        pings_within_1km=("within_1km", "sum")
    )
    .reset_index()
)

store_metrics["catchment_500m_pct"] = (
    store_metrics["pings_within_500m"]
    / store_metrics["total_mobility_pings"]
    * 100
)

store_metrics["catchment_1km_pct"] = (
    store_metrics["pings_within_1km"]
    / store_metrics["total_mobility_pings"]
    * 100
)

store_metrics = store_metrics.round(2)

store_metrics

,nearest_store_id,nearest_store_name,total_mobility_pings,avg_distance_km,pings_within_500m,pings_within_1km,catchment_500m_pct,catchment_1km_pct
0,S001,GeoCafe Central,316,0.63,101,288,31.96,91.14
1,S002,GeoCafe Market,199,0.65,64,174,32.16,87.44
2,S003,GeoCafe Riverside,536,1.48,89,269,16.60,50.19
3,S004,GeoCafe Station,602,1.91,68,193,11.30,32.06
4,S005,GeoCafe Downtown,702,1.85,76,242,10.83,34.47
5,S006,GeoCafe Mall,283,1.17,69,162,24.38,57.24
6,S007,GeoCafe University,517,1.24,91,299,17.60,57.83
7,S008,GeoCafe TechPark,648,1.74,68,235,10.49,36.27
8,S009,GeoCafe North,513,1.97,50,147,9.75,28.65
9,S010,GeoCafe South,684,2.02,43,159,6.29,23.25


In [15]:
store_metrics = store_metrics.merge(
    stores_df[
        [
            "store_id",
            "store_name",
            "store_type",
            "daily_capacity",
            "opening_date"
        ]
    ],
    left_on="nearest_store_id",
    right_on="store_id",
    how="left"
)

store_metrics = store_metrics.drop(
    columns=["store_id", "store_name"],
    errors="ignore"
)

store_metrics

,nearest_store_id,nearest_store_name,total_mobility_pings,avg_distance_km,pings_within_500m,pings_within_1km,catchment_500m_pct,catchment_1km_pct,store_type,daily_capacity,opening_date
0,S001,GeoCafe Central,316,0.63,101,288,31.96,91.14,High Street,450,2024-01-15
1,S002,GeoCafe Market,199,0.65,64,174,32.16,87.44,Market,400,2024-03-10
2,S003,GeoCafe Riverside,536,1.48,89,269,16.60,50.19,Riverside,350,2024-05-20
3,S004,GeoCafe Station,602,1.91,68,193,11.30,32.06,Transit,500,2024-07-01
4,S005,GeoCafe Downtown,702,1.85,76,242,10.83,34.47,Downtown,550,2024-09-15
5,S006,GeoCafe Mall,283,1.17,69,162,24.38,57.24,Shopping Mall,700,2025-01-10
6,S007,GeoCafe University,517,1.24,91,299,17.60,57.83,University,300,2025-02-15
7,S008,GeoCafe TechPark,648,1.74,68,235,10.49,36.27,Tech Park,600,2025-04-20
8,S009,GeoCafe North,513,1.97,50,147,9.75,28.65,High Street,400,2025-06-10
9,S010,GeoCafe South,684,2.02,43,159,6.29,23.25,Residential,350,2025-08-01


In [16]:
max_mobility = store_metrics["total_mobility_pings"].max()

store_metrics["mobility_intensity_score"] = (
    store_metrics["total_mobility_pings"]
    / max_mobility
    * 100
)

store_metrics["mobility_intensity_score"] = (
    store_metrics["mobility_intensity_score"].round(2)
)

store_metrics[
    [
        "nearest_store_id",
        "nearest_store_name",
        "total_mobility_pings",
        "mobility_intensity_score"
    ]
]

,nearest_store_id,nearest_store_name,total_mobility_pings,mobility_intensity_score
0,S001,GeoCafe Central,316,45.01
1,S002,GeoCafe Market,199,28.35
2,S003,GeoCafe Riverside,536,76.35
3,S004,GeoCafe Station,602,85.75
4,S005,GeoCafe Downtown,702,100.00
5,S006,GeoCafe Mall,283,40.31
6,S007,GeoCafe University,517,73.65
7,S008,GeoCafe TechPark,648,92.31
8,S009,GeoCafe North,513,73.08
9,S010,GeoCafe South,684,97.44


In [17]:
store_metrics["catchment_score"] = (
    store_metrics["catchment_500m_pct"] * 0.6
    + store_metrics["catchment_1km_pct"] * 0.4
)

store_metrics["catchment_score"] = (
    store_metrics["catchment_score"].round(2)
)

store_metrics[
    [
        "nearest_store_id",
        "nearest_store_name",
        "catchment_500m_pct",
        "catchment_1km_pct",
        "catchment_score"
    ]
]

,nearest_store_id,nearest_store_name,catchment_500m_pct,catchment_1km_pct,catchment_score
0,S001,GeoCafe Central,31.96,91.14,55.63
1,S002,GeoCafe Market,32.16,87.44,54.27
2,S003,GeoCafe Riverside,16.60,50.19,30.04
3,S004,GeoCafe Station,11.30,32.06,19.60
4,S005,GeoCafe Downtown,10.83,34.47,20.29
5,S006,GeoCafe Mall,24.38,57.24,37.52
6,S007,GeoCafe University,17.60,57.83,33.69
7,S008,GeoCafe TechPark,10.49,36.27,20.80
8,S009,GeoCafe North,9.75,28.65,17.31
9,S010,GeoCafe South,6.29,23.25,13.07


In [18]:
store_metrics["capacity_pressure_score"] = (
    store_metrics["total_mobility_pings"]
    / store_metrics["daily_capacity"]
    * 100
)

store_metrics["capacity_pressure_score"] = (
    store_metrics["capacity_pressure_score"].round(2)
)

store_metrics[
    [
        "nearest_store_id",
        "nearest_store_name",
        "total_mobility_pings",
        "daily_capacity",
        "capacity_pressure_score"
    ]
]

,nearest_store_id,nearest_store_name,total_mobility_pings,daily_capacity,capacity_pressure_score
0,S001,GeoCafe Central,316,450,70.22
1,S002,GeoCafe Market,199,400,49.75
2,S003,GeoCafe Riverside,536,350,153.14
3,S004,GeoCafe Station,602,500,120.40
4,S005,GeoCafe Downtown,702,550,127.64
5,S006,GeoCafe Mall,283,700,40.43
6,S007,GeoCafe University,517,300,172.33
7,S008,GeoCafe TechPark,648,600,108.00
8,S009,GeoCafe North,513,400,128.25
9,S010,GeoCafe South,684,350,195.43


In [19]:
def calculate_store_distance(row1, row2):
    return haversine_distance(
        row1["latitude"],
        row1["longitude"],
        row2["latitude"],
        row2["longitude"]
    )


store_pairs = []

for i in range(len(stores_df)):
    for j in range(i + 1, len(stores_df)):
        
        store_1 = stores_df.iloc[i]
        store_2 = stores_df.iloc[j]
        
        distance = calculate_store_distance(store_1, store_2)
        
        store_pairs.append({
            "store_1": store_1["store_id"],
            "store_2": store_2["store_id"],
            "distance_km": distance
        })

store_pairs_df = pd.DataFrame(store_pairs)

store_pairs_df.head(10)

,store_1,store_2,distance_km
0,S001,S002,1.438238
1,S001,S003,1.791297
2,S001,S004,2.015232
3,S001,S005,1.586754
4,S001,S006,2.222537
5,S001,S007,1.872708
6,S001,S008,2.177833
7,S001,S009,3.280250
8,S001,S010,3.529152
9,S002,S003,3.141890


In [20]:
close_store_pairs = store_pairs_df[
    store_pairs_df["distance_km"] <= 1.0
].copy()

close_store_pairs = close_store_pairs.sort_values(
    "distance_km"
)

print("Store pairs within 1 km:", len(close_store_pairs))

close_store_pairs

Store pairs within 1 km: 0


,store_1,store_2,distance_km


In [21]:
overlap_counts = {}

for store_id in stores_df["store_id"]:
    count = 0
    
    for _, pair in close_store_pairs.iterrows():
        if pair["store_1"] == store_id or pair["store_2"] == store_id:
            count += 1
    
    overlap_counts[store_id] = count

store_metrics["nearby_store_count"] = (
    store_metrics["nearest_store_id"]
    .map(overlap_counts)
    .fillna(0)
    .astype(int)
)

store_metrics[
    [
        "nearest_store_id",
        "nearest_store_name",
        "nearby_store_count"
    ]
]

,nearest_store_id,nearest_store_name,nearby_store_count
0,S001,GeoCafe Central,0
1,S002,GeoCafe Market,0
2,S003,GeoCafe Riverside,0
3,S004,GeoCafe Station,0
4,S005,GeoCafe Downtown,0
5,S006,GeoCafe Mall,0
6,S007,GeoCafe University,0
7,S008,GeoCafe TechPark,0
8,S009,GeoCafe North,0
9,S010,GeoCafe South,0


In [22]:
store_metrics["cannibalization_indicator"] = np.where(
    store_metrics["nearby_store_count"] >= 2,
    "High Overlap",
    np.where(
        store_metrics["nearby_store_count"] == 1,
        "Moderate Overlap",
        "Low Overlap"
    )
)

store_metrics[
    [
        "nearest_store_id",
        "nearest_store_name",
        "nearby_store_count",
        "cannibalization_indicator"
    ]
]

,nearest_store_id,nearest_store_name,nearby_store_count,cannibalization_indicator
0,S001,GeoCafe Central,0,Low Overlap
1,S002,GeoCafe Market,0,Low Overlap
2,S003,GeoCafe Riverside,0,Low Overlap
3,S004,GeoCafe Station,0,Low Overlap
4,S005,GeoCafe Downtown,0,Low Overlap
5,S006,GeoCafe Mall,0,Low Overlap
6,S007,GeoCafe University,0,Low Overlap
7,S008,GeoCafe TechPark,0,Low Overlap
8,S009,GeoCafe North,0,Low Overlap
9,S010,GeoCafe South,0,Low Overlap


In [23]:
store_metrics["opportunity_score"] = (
    store_metrics["mobility_intensity_score"] * 0.4
    + store_metrics["catchment_score"] * 0.3
    + store_metrics["capacity_pressure_score"] * 0.3
)

store_metrics["opportunity_score"] = (
    store_metrics["opportunity_score"]
    .clip(0, 100)
    .round(2)
)

store_metrics[
    [
        "nearest_store_id",
        "nearest_store_name",
        "mobility_intensity_score",
        "catchment_score",
        "capacity_pressure_score",
        "opportunity_score"
    ]
]

,nearest_store_id,nearest_store_name,mobility_intensity_score,catchment_score,capacity_pressure_score,opportunity_score
0,S001,GeoCafe Central,45.01,55.63,70.22,55.76
1,S002,GeoCafe Market,28.35,54.27,49.75,42.55
2,S003,GeoCafe Riverside,76.35,30.04,153.14,85.49
3,S004,GeoCafe Station,85.75,19.60,120.40,76.30
4,S005,GeoCafe Downtown,100.00,20.29,127.64,84.38
5,S006,GeoCafe Mall,40.31,37.52,40.43,39.51
6,S007,GeoCafe University,73.65,33.69,172.33,91.27
7,S008,GeoCafe TechPark,92.31,20.80,108.00,75.56
8,S009,GeoCafe North,73.08,17.31,128.25,72.90
9,S010,GeoCafe South,97.44,13.07,195.43,100.00


In [24]:
store_metrics = store_metrics.sort_values(
    "opportunity_score",
    ascending=False
).reset_index(drop=True)

store_metrics["opportunity_rank"] = (
    store_metrics.index + 1
)

store_metrics[
    [
        "opportunity_rank",
        "nearest_store_id",
        "nearest_store_name",
        "opportunity_score",
        "cannibalization_indicator"
    ]
]

,opportunity_rank,nearest_store_id,nearest_store_name,opportunity_score,cannibalization_indicator
0,1,S010,GeoCafe South,100.00,Low Overlap
1,2,S007,GeoCafe University,91.27,Low Overlap
2,3,S003,GeoCafe Riverside,85.49,Low Overlap
3,4,S005,GeoCafe Downtown,84.38,Low Overlap
4,5,S004,GeoCafe Station,76.30,Low Overlap
5,6,S008,GeoCafe TechPark,75.56,Low Overlap
6,7,S009,GeoCafe North,72.90,Low Overlap
7,8,S001,GeoCafe Central,55.76,Low Overlap
8,9,S002,GeoCafe Market,42.55,Low Overlap
9,10,S006,GeoCafe Mall,39.51,Low Overlap


In [25]:
final_store_analysis = store_metrics[
    [
        "opportunity_rank",
        "nearest_store_id",
        "nearest_store_name",
        "store_type",
        "total_mobility_pings",
        "avg_distance_km",
        "pings_within_500m",
        "pings_within_1km",
        "catchment_500m_pct",
        "catchment_1km_pct",
        "mobility_intensity_score",
        "catchment_score",
        "capacity_pressure_score",
        "nearby_store_count",
        "cannibalization_indicator",
        "opportunity_score"
    ]
].copy()

final_store_analysis

,opportunity_rank,nearest_store_id,nearest_store_name,store_type,total_mobility_pings,avg_distance_km,pings_within_500m,pings_within_1km,catchment_500m_pct,catchment_1km_pct,mobility_intensity_score,catchment_score,capacity_pressure_score,nearby_store_count,cannibalization_indicator,opportunity_score
0,1,S010,GeoCafe South,Residential,684,2.02,43,159,6.29,23.25,97.44,13.07,195.43,0,Low Overlap,100.00
1,2,S007,GeoCafe University,University,517,1.24,91,299,17.60,57.83,73.65,33.69,172.33,0,Low Overlap,91.27
2,3,S003,GeoCafe Riverside,Riverside,536,1.48,89,269,16.60,50.19,76.35,30.04,153.14,0,Low Overlap,85.49
3,4,S005,GeoCafe Downtown,Downtown,702,1.85,76,242,10.83,34.47,100.00,20.29,127.64,0,Low Overlap,84.38
4,5,S004,GeoCafe Station,Transit,602,1.91,68,193,11.30,32.06,85.75,19.60,120.40,0,Low Overlap,76.30
5,6,S008,GeoCafe TechPark,Tech Park,648,1.74,68,235,10.49,36.27,92.31,20.80,108.00,0,Low Overlap,75.56
6,7,S009,GeoCafe North,High Street,513,1.97,50,147,9.75,28.65,73.08,17.31,128.25,0,Low Overlap,72.90
7,8,S001,GeoCafe Central,High Street,316,0.63,101,288,31.96,91.14,45.01,55.63,70.22,0,Low Overlap,55.76
8,9,S002,GeoCafe Market,Market,199,0.65,64,174,32.16,87.44,28.35,54.27,49.75,0,Low Overlap,42.55
9,10,S006,GeoCafe Mall,Shopping Mall,283,1.17,69,162,24.38,57.24,40.31,37.52,40.43,0,Low Overlap,39.51


In [26]:
final_store_analysis.to_csv(
    "store_opportunity_analysis.csv",
    index=False
)

print("Saved: store_opportunity_analysis.csv")

Saved: store_opportunity_analysis.csv


In [27]:
print("========== DAY 6 VALIDATION ==========")

print("Stores analyzed:", len(final_store_analysis))

print(
    "Average opportunity score:",
    round(
        final_store_analysis["opportunity_score"].mean(),
        2
    )
)

print(
    "Highest opportunity score:",
    final_store_analysis["opportunity_score"].max()
)

print(
    "Store pairs within 1 km:",
    len(close_store_pairs)
)

print("\nOverlap distribution:")
print(
    final_store_analysis[
        "cannibalization_indicator"
    ].value_counts()
)

print("\nOutput file:")
print("- store_opportunity_analysis.csv")

========== DAY 6 VALIDATION ==========
Stores analyzed: 10
Average opportunity score: 72.37
Highest opportunity score: 100.0
Store pairs within 1 km: 0

Overlap distribution:
cannibalization_indicator
Low Overlap    10
Name: count, dtype: int64

Output file:
- store_opportunity_analysis.csv
